# 02 — Results: End-to-End Pipeline

This notebook runs the full fraud/AML detection pipeline:  
**Load Data → Feature Engineering → Isolation Forest → Risk Scoring → Explanations**

All outputs are saved to `reports/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.db import query
from src.features import build_features
from src.model import train_model, score_anomalies
from src.scoring import compute_risk_scores
from src.explain import generate_explanations

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

REPORTS = Path('../reports')
REPORTS.mkdir(exist_ok=True)

print('Imports ready.')

## Step 1 — Load Data

In [ ]:
txns = query('SELECT * FROM transactions')
accounts = query('SELECT * FROM accounts')

print(f'Transactions loaded: {len(txns):,}')
print(f'Accounts loaded:     {len(accounts):,}')

## Step 2 — Feature Engineering

In [ ]:
features = build_features(txns, accounts)
print(f'Feature matrix shape: {features.shape}')
print(f'Columns: {list(features.columns)}')
features.describe().round(2)

## Step 3 — Train Isolation Forest

In [ ]:
model, scaler = train_model(features, contamination=0.05)
print(f'Model trained with {model.n_estimators} trees.')
print(f'Contamination: {model.contamination}')

## Step 4 — Score Anomalies

In [ ]:
anomaly_df = score_anomalies(model, scaler, features)
print(f'Anomalies detected: {anomaly_df["is_anomaly"].sum()} / {len(anomaly_df)}')
anomaly_df.head()

## Step 5 — Risk Scoring

In [ ]:
risk_df = compute_risk_scores(anomaly_df)
print('Risk Band Distribution:')
print(risk_df['risk_band'].value_counts())
print(f'\nRisk score stats:')
print(risk_df['risk_score'].describe().round(1))

---
## Visualization 1 — Risk Score Distribution

Most accounts cluster at low risk, with a tail of high-risk outliers flagged by the Isolation Forest.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colors = {'Low': '#4CAF50', 'Medium': '#FFC107', 'High': '#F44336'}

for band, color in colors.items():
    subset = risk_df[risk_df['risk_band'] == band]
    ax.hist(subset['risk_score'], bins=30, alpha=0.75, label=f'{band} ({len(subset)})',
            color=color, edgecolor='white', linewidth=0.5)

ax.axvline(40, color='orange', linestyle='--', linewidth=1, alpha=0.7)
ax.axvline(70, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax.set_title('Risk Score Distribution Across All Accounts')
ax.set_xlabel('Risk Score (0–100)')
ax.set_ylabel('Number of Accounts')
ax.legend(title='Risk Band')

plt.tight_layout()
plt.savefig(REPORTS / 'risk_score_distribution.png', bbox_inches='tight')
plt.show()

## Step 6 — Generate Explanations for High-Risk Accounts

In [ ]:
explained = generate_explanations(risk_df, features, min_risk_score=70.0)
print(f'High-risk accounts explained: {len(explained)}')

---
## Visualization 2 — Top 20 Highest-Risk Accounts with Explanations

In [ ]:
top20 = explained.nlargest(20, 'risk_score')[['account_id', 'risk_score', 'risk_band', 'explanation']].copy()
top20['account_id'] = top20['account_id'].str[:12] + '...'
top20 = top20.reset_index(drop=True)
top20.index = top20.index + 1
top20.index.name = 'Rank'

# Display styled table
display(top20.style.set_properties(**{
    'text-align': 'left',
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]},
]).bar(subset=['risk_score'], color='#FF6B6B', vmin=0, vmax=100))

# Also save as CSV
top20.to_csv(REPORTS / 'top20_high_risk_accounts.csv')
print(f'\nSaved to reports/top20_high_risk_accounts.csv')

---
## Visualization 3 — Feature Importance (Permutation-based Proxy)

Since Isolation Forest doesn't directly expose feature importances, we use a **variance-based proxy**: features with higher variance in their contribution to anomaly scores carry more weight in the model's decisions.

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler

# Use the trained scaler and model to compute permutation importance
X_scaled = scaler.transform(features)

# Use anomaly_score as the "target" for permutation importance
# We compute it based on how much each feature's permutation degrades the score
perm_result = permutation_importance(
    model, X_scaled, model.decision_function(X_scaled),
    n_repeats=10, random_state=42, n_jobs=-1
)

importance_df = pd.DataFrame({
    'feature': features.columns,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std,
}).sort_values('importance_mean', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['feature'], importance_df['importance_mean'],
        xerr=importance_df['importance_std'],
        color=sns.color_palette('viridis', len(importance_df)),
        edgecolor='white', linewidth=0.5)
ax.set_title('Feature Importance (Permutation-based)')
ax.set_xlabel('Mean Importance')

plt.tight_layout()
plt.savefig(REPORTS / 'feature_importance.png', bbox_inches='tight')
plt.show()

---
## Visualization 4 — Feature Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = features.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.savefig(REPORTS / 'feature_correlation.png', bbox_inches='tight')
plt.show()

---
## Visualization 5 — Risk Band Breakdown

In [ ]:
band_counts = risk_df['risk_band'].value_counts()
colors_pie = ['#4CAF50', '#FFC107', '#F44336']

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    band_counts.values, labels=band_counts.index, autopct='%1.1f%%',
    colors=colors_pie, startangle=90, textprops={'fontsize': 12}
)
ax.set_title('Account Risk Band Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(REPORTS / 'risk_band_pie.png', bbox_inches='tight')
plt.show()

print('\n=== Pipeline Complete ===')
print(f'Total accounts scored: {len(risk_df)}')
print(f'High-risk accounts:    {len(explained)}')
print(f'Charts saved to:       reports/')